# Graded Lab: Agentic Workflows

In this lab, you will build an agentic system that generates a short research report through planning, external tool usage, and feedback integration. Your workflow will involve:

### Agents

* **Planning Agent / Writer**: Creates an outline and coordinates tasks.
* **Research Agent**: Gathers external information using tools like Arxiv, Tavily, and Wikipedia.
* **Editor Agent**: Reflects on the report and provides suggestions for improvement.

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to write your solution code or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the <span style="background-color: red; color: white; padding: 3px 5px; font-size: 16px; border-radius: 5px;">Submit assignment</span> button on the top right of the page.
---


### Research Tools

By importing `research_tools`, you gain access to several search utilities:

- `research_tools.arxiv_search_tool(query)` → search academic papers from **arXiv**  

  *Example:* `research_tools.arxiv_search_tool("neural networks for climate modeling")`

- `research_tools.tavily_search_tool(query)` → perform web searches with the **Tavily API**  

  *Example:* `research_tools.tavily_search_tool("latest trends in sunglasses fashion")`

- `research_tools.wikipedia_search_tool(query)` → retrieve summaries from **Wikipedia**  

  *Example:* `research_tools.wikipedia_search_tool("Ensemble Kalman Filter")`

Run the cell below to make them available.

In [4]:
# =========================
# Imports
# =========================

# --- Standard library 
from datetime import datetime
import re
import json
import ast


# --- Third-party ---
from IPython.display import Markdown, display
from aisuite import Client

# --- Local / project ---
import research_tools

In [5]:
import unittests

### Initialize client

Create a shared client instance for upcoming calls.

In [6]:
CLIENT = Client()

## Exercise 1: planner_agent

### Objective
Correctly set up a call to a language model (LLM) to generate a research plan.

### Instructions

1. **Focus Areas**:
   - Ensure `CLIENT.chat.completions.create` is correctly configured.
   - Pass the `model` and `messages` parameters correctly:
     - **Model**: Use `"openai:o4-mini"` by default.
     - **Messages**: Set with `{"role": "user", "content": user_prompt}`.
     - **Temperature**: Fixed at 1 for creative outputs.

### Notes

- The prompt is pre-defined and guides the LLM on task requirements.
- Only return a formatted list of steps — no extra text.

Focus on the LLM call setup to complete the task.

In [7]:
# GRADED FUNCTION: planner_agent

def planner_agent(topic: str, model: str = "openai:o4-mini") -> list[str]:
    """
    Generates a plan as a Python list of steps (strings) for a research workflow.

    Args:
        topic (str): Research topic to investigate.
        model (str): Language model to use.

    Returns:
        List[str]: A list of executable step strings.
    """

    
    # Build the user prompt
    user_prompt = f"""
    You are a planning agent responsible for organizing a research workflow with multiple intelligent agents.

    🧠 Available agents:
    - A research agent who can search the web, Wikipedia, and arXiv.
    - A writer agent who can draft research summaries.
    - An editor agent who can reflect and revise the drafts.

    🎯 Your job is to write a clear, step-by-step research plan **as a valid Python list**, where each step is a string.
    Each step should be atomic, executable, and must rely only on the capabilities of the above agents.

    🚫 DO NOT include irrelevant tasks like "create CSV", "set up a repo", "install packages", etc.
    ✅ DO include real research-related tasks (e.g., search, summarize, draft, revise).
    ✅ DO assume tool use is available.
    ✅ DO NOT include explanation text — return ONLY the Python list.
    ✅ The final step should be to generate a Markdown document containing the complete research report.

    Topic: "{topic}"
    """

    # Add the user prompt to the messages list
    messages = [{"role": "user", "content": user_prompt}]

    ### START CODE HERE ###

    # Call the LLM
    response = CLIENT.chat.completions.create( 
        # Pass in the model
        model="openai:o4-mini",
        # Define the messages. Remember this is meant to be a user prompt!
        messages=[{"role": "user", "content": user_prompt}],
        # Keep responses creative
        temperature=1, 
    )

    ### END CODE HERE ###

    # Extract message from response
    steps_str = response.choices[0].message.content.strip()

    # Parse steps
    steps = ast.literal_eval(steps_str)

    return steps

In [8]:
# Test your code!
unittests.test_planner_agent(planner_agent)

 All tests passed!


## Exercise 2: research_agent

### Objective
Set up a call to a language model (LLM) to perform a research task using various tools.

### Instructions

**Focus Areas**:

- **Creating a Custom Prompt**:
  - **Define the Role**: Clearly specify the role, such as "research assistant."
  - **List Available Tools** (as strings inside the prompt, not the actual functions):
    - Use `arxiv_tool` to find academic papers.
    - Use `tavily_tool` for general web searches.
    - Use `wikipedia_tool` for accessing encyclopedic knowledge.
  - **Specify the Task**: Include a placeholder in your prompt for defining the specific task that needs to be accomplished.
  - **Include Date Information**: Add a placeholder for the current date or time to provide context.

- **Creating Messages Dict**:
  - Ensure the `messages` are correctly set with `{"role": "user", "content": prompt}`.

- **Creating Tools List**:
  - Create a list of tools for use, such as `research_tools.arxiv_search_tool`, `research_tools.tavily_search_tool`, and `research_tools.wikipedia_search_tool`.

- **Correctly Setting the Call to the LLM**:
  - Pass the `model`, `messages`, and `tools` parameters accurately.
  - Set `tool_choice` to `"auto"` for automatic tool selection.
  - Limit interactions with `max_turns=6`.

### Notes

- The function provides pre-coded blocks where you need to replace placeholder values.
- The approach allows the LLM to use tools dynamically based on the task.

Focus on accurately setting the messages, tools, and LLM call parameters to complete the task.

In [13]:
# GRADED FUNCTION: research_agent

def research_agent(task: str, model: str = "openai:gpt-4o", return_messages: bool = False):
    """
    Executes a research task using tools via aisuite (no manual loop).
    Returns either the assistant text, or (text, messages) if return_messages=True.
    """
    print("==================================")  
    print("🔍 Research Agent")                 
    print("==================================")

    current_time = datetime.now().strftime('%Y-%m-%d')
    
    ### START CODE HERE ###

    # Create a customizable prompt by defining the role (e.g., "research assistant"),
    # listing tools (arxiv_tool, tavily_tool, wikipedia_tool) for various searches,
    # specifying the task with a placeholder, and including a current_time placeholder.
    prompt = \
    f"""
    You are a research assistant that can search the web and arXiv to write detailed,
    accurate, and properly sourced research reports
    Use tools when appropriate: arxiv_tool, tavily_tool, wikipedia_tool
    Cite sources whenever relevant. Do NOT omit citations for brevity
    When possible, include full URLs (arXiv links, web sources, etc.)
    Use an academic tone, organize output into clearly labeled sections, and include
    inline citations or footnotes as needed.
    Do not include placeholder text such as '(citation needed)' or '(citations omitted)'
    
    topic of research: {task}
    current date for reference: {current_time}
    """
    
    # Create the messages dict to pass to the LLM. Remember this is a user prompt!
    messages = [{"role": "user", "content": prompt}]

    # Save all of your available tools in the tools list. These can be found in the research_tools module.
    # You can identify each tool in your list like this: 
    # research_tools.<name_of_tool>, where <name_of_tool> is replaced with the function name of the tool.
    tools = [research_tools.arxiv_search_tool, research_tools.tavily_search_tool, research_tools.wikipedia_search_tool]
    
    # Call the model with tools enabled
    response = CLIENT.chat.completions.create(  
        # Set the model
        model=model,
        # Pass in the messages. You already defined this!
        messages=messages,
        # Pass in the tools list. You already defined this!
        tools=tools,
        # Set the LLM to automatically choose the tools
        tool_choice="auto",
        # Set the max turns to 6
        max_turns=6
    )  
    
    ### END CODE HERE ###

    content = response.choices[0].message.content
    print("✅ Output:\n", content)

    
    return (content, messages) if return_messages else content  

In [14]:
# Test your code!
unittests.test_research_agent(research_agent)

🔍 Research Agent
✅ Output:
 # Research Report: Key References for 2026

This report summarizes the latest references and emerging research trends for 2026 as derived from academic articles, web articles, and other credible sources.

## ArXiv References

1. **ICASSP 2026 Automatic Song Aesthetics Evaluation Challenge**

   This paper summarizes the ICASSP 2026 challenge which is focused on predicting the aesthetic scores of AI-generated songs. It involves two tracks aimed at predicting overall musicality and five fine-grained aesthetic scores. The challenge demonstrated significant advancements in aligning objective metrics with human aesthetic preferences and has established a standardized benchmark for evaluating modern music generation systems ([source](http://arxiv.org/abs/2601.07237v1), [PDF](https://arxiv.org/pdf/2601.07237v1)).

2. **ICASSP 2026 URGENT Speech Enhancement Challenge**

   This research outlines the ICASSP 2026 URGENT Challenge, emphasizing the development of univer

## Exercise 3: writer_agent

### Objective
Set up a call to a language model (LLM) for executing writing tasks like drafting, expanding, or summarizing text.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of a writing agent focused on generating academic or technical content.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The function is designed to produce well-structured text by setting the correct prompts.
- Temperature is set to 1.0 to allow for creative variance in the writing outputs.

Ensure the system prompt and messages are defined properly to achieve a structured output from the LLM.

In [15]:
# GRADED FUNCTION: writer_agent
def writer_agent(task: str, model: str = "openai:gpt-4o") -> str: # @REPLACE def writer_agent(task: str, model: str = None) -> str:
    """
    Executes writing tasks, such as drafting, expanding, or summarizing text.
    """
    print("==================================")
    print("✍️ Writer Agent")
    print("==================================")

    ### START CODE HERE ###
    
    # Create the system prompt.
    # This should assign the LLM the role of a writing agent specialized in generating well-structured academic or technical content
    
    system_prompt = \
    """
    You are a writing agent specialized in generating well-structured academic or technical content.
    You excel in tasks  like drafting, expanding, or summarizing text.
    write for the topic below:
    """

    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = [{"role": "system", "content": system_prompt}]

    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = [{"role": "user", "content": task}]

    # Add both system and user messages to the messages list
    messages = system_msg + user_msg
    
    ### END CODE HERE ###

    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=1.0
    )

    return response.choices[0].message.content

In [16]:
# Test your code!
unittests.test_writer_agent(writer_agent)

✍️ Writer Agent
 All tests passed!


## Exercise 4: editor_agent

### Objective
Configure a call to a language model (LLM) to perform editorial tasks such as reflecting, critiquing, or revising drafts.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of an editor agent whose task is to reflect on, critique, or improve drafts.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The editor agent is tailored for enhancing the quality of text by setting an appropriate role and task in the prompts.
- Temperature is set to 0.7, balancing creativity and coherence in editorial outputs.

Ensure the system prompt and messages are accurately set up to perform effective editorial tasks with the LLM.

In [17]:
# GRADED FUNCTION: editor_agent
def editor_agent(task: str, model: str = "openai:gpt-4o") -> str:
    """
    Executes editorial tasks such as reflection, critique, or revision.
    """
    print("==================================")
    print("🧠 Editor Agent")
    print("==================================")
    
    ### START CODE HERE ###

    # Create the system prompt.
    # This should assign the LLM the role of an editor agent specialized in reflecting on, critiquing, or improving existing drafts.
    system_prompt = \
    """
    You are an editor agent whose task is to reflect on, critique, or improve research drafts.
    Generates a structured reflection AND a revised research report.
    Returns:
        dict with keys:
          - "reflection": structured reflection text
          - "revised_report": improved version of the input report

    """
    
    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = [{"role": "system", "content": system_prompt}]

    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = [{"role": "user", "content": task}]

    # Add both system and user messages to the messages list
    messages = system_msg + user_msg
        
    ### END CODE HERE ###
    
    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=0.7 
    )
    
    return response.choices[0].message.content

In [18]:
# Test your code!
unittests.test_editor_agent(editor_agent)

🧠 Editor Agent
 All tests passed!


### 🎯 The Executor Agent

The `executor_agent` manages the workflow by executing each step of a given plan. It:

1. Decides **which agent** (`research_agent`, `writer_agent`, or `editor_agent`) should handle the step.
2. Builds context from the outputs of previous steps.
3. Sends the enriched task to the selected agent.
4. Collects and stores the results in a shared history.

👉 **Do not implement or modify this function.** It is already provided as the orchestration component of the multi-agent pipeline.

Notice that `planner_agent` might return a long list of steps. Because of this, the maximum number of steps is set to a maximum of 4 to keep running time reasonable.

In [19]:
agent_registry = {
    "research_agent": research_agent,
    "editor_agent": editor_agent,
    "writer_agent": writer_agent,
}

def clean_json_block(raw: str) -> str:
    """
    Clean the contents of a JSON block that may come wrapped with Markdown backticks.
    """
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return raw.strip()

In [20]:
def executor_agent(topic, model: str = "openai:gpt-4o", limit_steps: bool = True):

    plan_steps = planner_agent(topic)
    max_steps = 4

    if limit_steps:
        plan_steps = plan_steps[:min(len(plan_steps), max_steps)]
    
    history = []

    print("==================================")
    print("🎯 Editor Agent")
    print("==================================")

    for i, step in enumerate(plan_steps):

        agent_decision_prompt = f"""
        You are an execution manager for a multi-agent research team.

        Given the following instruction, identify which agent should perform it and extract the clean task.

        Return only a valid JSON object with two keys:
        - "agent": one of ["research_agent", "editor_agent", "writer_agent"]
        - "task": a string with the instruction that the agent should follow

        Only respond with a valid JSON object. Do not include explanations or markdown formatting.

        Instruction: "{step}"
        """
        response = CLIENT.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": agent_decision_prompt}],
            temperature=0,
        )

        raw_content = response.choices[0].message.content
        cleaned_json = clean_json_block(raw_content)
        agent_info = json.loads(cleaned_json)

        agent_name = agent_info["agent"]
        task = agent_info["task"]

        context = "\n".join([
            f"Step {j+1} executed by {a}:\n{r}" 
            for j, (s, a, r) in enumerate(history)
        ])
        enriched_task = f"""
        You are {agent_name}.

        Here is the context of what has been done so far:
        {context}

        Your next task is:
        {task}
        """

        print(f"\n🛠️ Executing with agent: `{agent_name}` on task: {task}")

        if agent_name in agent_registry:
            output = agent_registry[agent_name](enriched_task)
            history.append((step, agent_name, output))
        else:
            output = f"⚠️ Unknown agent: {agent_name}"
            history.append((step, agent_name, output))

        print(f"✅ Output:\n{output}")

    return history

In [22]:
plan_steps = planner_agent("Agentic AI")

In [23]:
plan_steps

['Use the research agent to perform a web search for foundational definitions and surveys on agentic AI',
 'Use the research agent to query Wikipedia for an overview and key concepts related to agentic AI',
 'Use the research agent to search arXiv for recent papers on agentic AI frameworks and methodologies',
 'Use the research agent to compile a list of relevant publications and their abstracts on agentic AI',
 'Use the writer agent to draft a literature review section based on the compiled abstracts and publications',
 'Use the writer agent to draft a definitions section summarizing key concepts of agentic AI',
 'Use the writer agent to draft a methodology section outlining common experimental setups in agentic AI research',
 'Use the writer agent to draft a discussion section analyzing trends and open challenges in agentic AI',
 'Use the editor agent to review and revise the literature review and definitions sections for clarity and coherence',
 'Use the editor agent to review and r

In [21]:
# If you want to see the full workflow without limiting the number of steps. Set limit_steps to False
# Keep in mind this could take more than 10 minutes to complete
executor_history = executor_agent("The ensemble Kalman filter for time series forecasting", limit_steps=True)

md = executor_history[-1][-1].strip("`")  
display(Markdown(md))

🎯 Editor Agent

🛠️ Executing with agent: `research_agent` on task: Search Wikipedia, arXiv, and the web for fundamental literature on the ensemble Kalman filter and its applications to time series forecasting
🔍 Research Agent
✅ Output:
 ## Introduction

The Ensemble Kalman Filter (EnKF) is a recursive filtering method tailored for large-scale problems, commonly used in geophysical models that involve discretizations of partial differential equations. It originated as an adaptation of the classical Kalman filter for high-dimensional systems, where the ensemble mean and covariance are computed from a sample of state vectors. The primary advantage of EnKF is its efficiency in handling cases where the underlying probability distributions can be approximated as Gaussian.[^1^]

### Functionality and Characteristics

EnKF is a Monte Carlo approach to the Bayesian update problem, where it utilizes Bayes' theorem to update the probability distribution function (PDF) of the system state based on


🛠️ Executing with agent: `research_agent` on task: Extract abstracts, key algorithmic details, and performance metrics from the top 15 most relevant papers
🔍 Research Agent
✅ Output:
 ## Abstracts and Key Details of Relevant ArXiv Papers on Ensemble Kalman Filter

### Paper 1: Ensemble Kalman Filtering Meets Gaussian Process SSM for Non-Mean-Field and Online Inference
- **Authors**: Zhidi Lin, Yiyong Sun, Feng Yin, Alexandre Hoang Thiéry
- **Published**: December 10, 2023
- **Abstract**: This paper explores the integration of the Ensemble Kalman Filter (EnKF) with Gaussian process state-space models (GPSSMs) under a non-mean-field assumption, addressing limitations in variational inference. The proposed method combines EnKF to improve inference accuracy and operational feasibility in online learning. It demonstrates significant performance enhancements against existing methods across diverse datasets.
- **Key Algorithmic Details**: Incorporates EnKF into variational inference for more


🛠️ Executing with agent: `research_agent` on task: Compile and structure mathematical formulations and pseudocode of the standard ensemble Kalman filter
🔍 Research Agent
✅ Output:
 ## Mathematical Formulations and Pseudocode of the Standard Ensemble Kalman Filter

The Ensemble Kalman Filter (EnKF) utilizes a Monte Carlo method-based approach to linear Kalman filtering. While it primarily addresses the same problem space as the classic Kalman filter, EnKF extends its applicability to high-dimensional, nonlinear systems by leveraging an ensemble of forecasts or states to estimate the system's state.

### Mathematical Formulation of the Standard Ensemble Kalman Filter

Given:
- **State Vector**: \(\mathbf{x}_k \in \mathbb{R}^n\) at time \(k\)
- **Observation Vector**: \(\mathbf{y}_k \in \mathbb{R}^m\)
- **State Transition Model**: \(\mathbf{x}_{k+1} = \mathbf{f}(\mathbf{x}_k) + \mathbf{w}_k\), where \(\mathbf{w}_k\) is process noise
- **Observation Model**: \(\mathbf{y}_k = \mathbf{h}(\m


🛠️ Executing with agent: `research_agent` on task: Identify and summarize at least three variants of the ensemble Kalman filter used in time series forecasting, including their reported comparative performance.
🔍 Research Agent
✅ Output:
 ## Variants of the Ensemble Kalman Filter for Time Series Forecasting

The exploration of variants of the Ensemble Kalman Filter (EnKF) within the domain of time series forecasting has led to significant advancements in predictive accuracy and computational efficiency. The following sections detail several notable variants, their methodological innovations, and comparative performance metrics.

### Variants and Their Innovations

#### 1. Ensemble Kalman Filtering Meets Gaussian Process State-Space Models (GPSSMs)

- **Authors**: Zhidi Lin, Yiyong Sun, Feng Yin, Alexandre Hoang Thiéry
- **Published**: December 10, 2023
- **Summary**: This variant integrates EnKF with Gaussian process state-space models under a non-mean-field assumption. It addresses v

## Variants of the Ensemble Kalman Filter for Time Series Forecasting

The exploration of variants of the Ensemble Kalman Filter (EnKF) within the domain of time series forecasting has led to significant advancements in predictive accuracy and computational efficiency. The following sections detail several notable variants, their methodological innovations, and comparative performance metrics.

### Variants and Their Innovations

#### 1. Ensemble Kalman Filtering Meets Gaussian Process State-Space Models (GPSSMs)

- **Authors**: Zhidi Lin, Yiyong Sun, Feng Yin, Alexandre Hoang Thiéry
- **Published**: December 10, 2023
- **Summary**: This variant integrates EnKF with Gaussian process state-space models under a non-mean-field assumption. It addresses variational inference challenges in GPSSMs by streamlining parameterization and enabling closed-form approximations of the evidence lower bound (ELBO). This integration improves inference accuracy and operational feasibility for online learning applications[^1^].
- **Performance**: The proposed framework demonstrates superior learning and inference across diverse datasets, outperforming existing methods in terms of accuracy and efficiency.
- **URL**: [arXiv](http://arxiv.org/abs/2312.05910v5)

#### 2. LLM-Mixer: Multiscale Mixing in LLMs for Time Series Forecasting

- **Authors**: Md Kowsher, Md. Shohanur Islam Sobuj, Nusrat Jahan Prottasha, E. Alejandro Alanis, Ozlem Ozmen Garibay, Niloofar Yousefi
- **Published**: October 15, 2024
- **Summary**: LLM-Mixer improves forecasting by combining multiscale time-series decomposition with pre-trained Large Language Models (LLMs). The data is decomposed into multiple temporal resolutions, processed by a frozen LLM tailored for time-series data. This approach captures both short-term fluctuations and long-term trends.
- **Performance**: Extensive experiments indicate LLM-Mixer's competitive performance, surpassing state-of-the-art models across various forecasting horizons[^2^].
- **URL**: [arXiv](http://arxiv.org/abs/2410.11674v2)

#### 3. A Variant of the Kalman Filter with ARMA Gaussian Noises

- **Authors**: Song Fang, Quanyan Zhu
- **Published**: January 9, 2020
- **Summary**: This variant deals with feedback channels and ARMA Gaussian noises. It provides explicit lower bounds on feedback capacity and recursive coding schemes, contributing insights into feedback coding design.
- **Performance**: The results not only align with existing theoretical frameworks but suggest potentially tighter bounds for feedback capacity[^3^].
- **URL**: [arXiv](http://arxiv.org/abs/2001.03108v6)

### Comparative Performance

1. **Inference Accuracy**: The integration of EnKF with GPSSMs significantly enhances inference accuracy, minimizing extensive parameterization challenges.
2. **Scalability**: Both LLM-Mixer and the GPSSM variant demonstrate enhanced scalability, crucial for handling large-scale time series data.
3. **Algorithmic Efficiency**: The recursive coding design in the ARMA variant of the Kalman filter provides a pragmatic approach for efficient feedback scenarios.

### Conclusion

The development and optimization of Ensemble Kalman Filter variants illustrate the dynamic capability of these methods in improving time-series forecasting. By incorporating advanced statistical models and leveraging machine learning advancements, these variants continue to redefine forecasting potential in complex data environments.

---

[^1^]: [Integration of EnKF with GPSSMs](https://arxiv.org/pdf/2312.05910v5)
[^2^]: [Link to LLM-Mixer on arXiv](https://arxiv.org/pdf/2410.11674v2)
[^3^]: [ARMA Gaussian Noises in Kalman Filter](https://arxiv.org/pdf/2001.03108v6)

## Check grading feedback

If you have collapsed the right panel to have more screen space for your code, as shown below:

<img src="./images/collapsed.png" alt="Collapsed Image" width="800" height="400"/>

You can click on the left-facing arrow button (highlighted in red) to view feedback for your submission after submitting it for grading. Once expanded, it should display like this:

<img src="./images/expanded.png" alt="Expanded Image" width="800" height="400"/>